In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft
from ipywidgets import interact, FloatSlider, IntSlider

def advanced_positive_lfm_db():
    """
    Продвинутый анализ ЛЧМ сигнала с новыми параметрами
    """
    
    def plot_lfm(T=1.0, f_start=1000, f_end=2000):
        # Автоматическая частота дискретизации
        Fs = f_end * 2.5
        
        # Генерация сигнала
        t = np.linspace(0, T, int(Fs * T), endpoint=False)
        k = (f_end - f_start) / T  # Коэффициент ЛЧМ модуляции (МГц/мс)
        deviation = abs(f_end - f_start)  # Девиация частоты (МГц)
        phase = 2 * np.pi * (f_start * t + 0.5 * k * t**2)
        signal = np.cos(phase)
        
        # Спектр - только положительные частоты
        n = len(signal)
        freq = np.fft.fftfreq(n, 1/Fs)
        spectrum = fft(signal)
        amp_spectrum = np.abs(spectrum) / n
        amp_spectrum_db = 20 * np.log10(amp_spectrum + 1e-12)
        
        # Положительные частоты
        positive_mask = freq >= 0
        freq_pos = freq[positive_mask]
        amp_db_pos = amp_spectrum_db[positive_mask]
        amp_linear_pos = amp_spectrum[positive_mask]
        
        # Находим максимум и уровень -3 дБ
        max_amp_db = np.max(amp_db_pos)
        max_amp_linear = np.max(amp_linear_pos)
        threshold_db = max_amp_db - 3
        
        # Находим индекс максимума
        max_idx = np.argmax(amp_db_pos)
        max_freq = freq_pos[max_idx]
        
        # Ищем границы -3 дБ слева и справа от максимума
        f_low_3db = freq_pos[0]
        f_high_3db = freq_pos[-1]
        
        # Левая граница (от максимума вниз до -3 дБ)
        for i in range(max_idx, 0, -1):
            if amp_db_pos[i] <= threshold_db:
                if i < len(amp_db_pos) - 1:
                    x1, x2 = freq_pos[i], freq_pos[i+1]
                    y1, y2 = amp_db_pos[i], amp_db_pos[i+1]
                    f_low_3db = x1 + (threshold_db - y1) * (x2 - x1) / (y2 - y1)
                else:
                    f_low_3db = freq_pos[i]
                break
        
        # Правая граница (от максимума вверх до -3 дБ)
        for i in range(max_idx, len(amp_db_pos)):
            if amp_db_pos[i] <= threshold_db:
                if i > 0:
                    x1, x2 = freq_pos[i-1], freq_pos[i]
                    y1, y2 = amp_db_pos[i-1], amp_db_pos[i]
                    f_high_3db = x1 + (threshold_db - y1) * (x2 - x1) / (y2 - y1)
                else:
                    f_high_3db = freq_pos[i]
                break
        
        bandwidth_3db = f_high_3db - f_low_3db
        
        # Расчет крутизны спада на граничных частотах (в октавах)
        def find_amplitude_at_frequency(target_freq, freq_array, amp_db_array):
            """Находит амплитуду на заданной частоте с помощью интерполяции"""
            # Ищем ближайшие точки для интерполяции
            idx = np.searchsorted(freq_array, target_freq)
            if idx == 0:
                return amp_db_array[0]
            elif idx == len(freq_array):
                return amp_db_array[-1]
            else:
                # Линейная интерполяция
                x1, x2 = freq_array[idx-1], freq_array[idx]
                y1, y2 = amp_db_array[idx-1], amp_db_array[idx]
                return y1 + (target_freq - x1) * (y2 - y1) / (x2 - x1)
        
        # Точки для расчета крутизны спада
        f_low_half = f_low_3db / 2      # На октаву ниже нижней границы
        f_high_double = f_high_3db * 2  # На октаву выше верхней границы
        
        # Амплитуды в этих точках
        amp_low_3db = find_amplitude_at_frequency(f_low_3db, freq_pos, amp_db_pos)
        amp_low_half = find_amplitude_at_frequency(f_low_half, freq_pos, amp_db_pos)
        amp_high_3db = find_amplitude_at_frequency(f_high_3db, freq_pos, amp_db_pos)
        amp_high_double = find_amplitude_at_frequency(f_high_double, freq_pos, amp_db_pos)
        
        # Крутизна спада (дБ/октава)
        # Для нижней границы: от f_low до f_low/2 (вниз на октаву)
        slope_low = amp_low_3db - amp_low_half
        
        # Для верхней границы: от f_high до f_high*2 (вверх на октаву)
        slope_high = amp_high_3db - amp_high_double
        
        # Графики
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
        
        # Временная область
        ax1.plot(t, signal, 'b-', linewidth=1)
        ax1.set_xlabel('Время (мс)')
        ax1.set_ylabel('Амплитуда')
        ax1.set_title(f'ЛЧМ сигнал: {f_start} → {f_end} МГц\nT = {T} мс, k = {k:.0f} МГц/мс')
        ax1.grid(True, alpha=0.3)
        
        # Частотная область
        ax2.plot(freq_pos, amp_db_pos, 'r-', linewidth=2, label='Спектр ЛЧМ')
        
        # Уровень -3 дБ и границы полосы
        ax2.axhline(threshold_db, color='green', linestyle='--', linewidth=2, label='Уровень -3 дБ')
        ax2.axvline(f_low_3db, color='purple', linestyle='-', linewidth=2, label=f'f_low = {f_low_3db:.0f} МГц')
        ax2.axvline(f_high_3db, color='purple', linestyle='-', linewidth=2, label=f'f_high = {f_high_3db:.0f} МГц')
        
        # Заливка полосы пропускания
        ax2.fill_betweenx([amp_db_pos.min(), max_amp_db], f_low_3db, f_high_3db, 
                         alpha=0.15, color='purple', label=f'BW = {bandwidth_3db:.0f} МГц')
        
        # Точки для расчета крутизны спада
        ax2.plot(f_low_half, amp_low_half, 'bo', markersize=8, label=f'f_low/2 = {f_low_half:.0f} МГц')
        ax2.plot(f_high_double, amp_high_double, 'go', markersize=8, label=f'f_high×2 = {f_high_double:.0f} МГц')
        
        ax2.set_xlabel('Частота (МГц)')
        ax2.set_ylabel('Амплитуда (дБ)')
        ax2.set_title('Спектр ЛЧМ сигнала с анализом крутизны спада')
        ax2.legend(loc='upper right')
        ax2.grid(True, alpha=0.3)
        
        # Настройка осей
        x_min = max(0, min(f_start, f_low_3db, f_low_half) - 500)
        x_max = max(f_end, f_high_3db, f_high_double) + 500
        ax2.set_xlim([x_min, x_max])
        ax2.set_ylim([max(amp_db_pos.min(), -80), max_amp_db + 5])
        
        plt.tight_layout()
        plt.show()
        
        # Вывод подробной информации
        print(f"🔧 ОСНОВНЫЕ ПАРАМЕТРЫ ЛЧМ:")
        print(f"   • Коэффициент ЛЧМ (k):    {k:>8.0f} МГц/мс")
        print(f"   • Девиация частоты:       {deviation:>8.0f} МГц")
        print(f"   • Длительность:           {T:>8.1f} мс")
        print(f"   • Начальная частота:      {f_start:>8.0f} МГц")
        print(f"   • Конечная частота:       {f_end:>8.0f} МГц")
        
        print(f"\n📊 АНАЛИЗ ПОЛОСЫ ПРОПУСКАНИЯ:")
        print(f"   • Полоса по настройкам:   {f_end - f_start:>8.0f} МГц")
        print(f"   • Полоса по -3 дБ:        {bandwidth_3db:>8.0f} МГц")
        print(f"   • Нижняя граница (-3 дБ): {f_low_3db:>8.0f} МГц")
        print(f"   • Верхняя граница (-3 дБ): {f_high_3db:>8.0f} МГц")
        
        print(f"\n📉 КРУТИЗНА СПАДА (дБ/октава):")
        print(f"   Нижняя граница:")
        print(f"     • f_low = {f_low_3db:.0f} МГц: {amp_low_3db:.1f} дБ")
        print(f"     • f_low/2 = {f_low_half:.0f} МГц: {amp_low_half:.1f} дБ")
        print(f"     • Крутизна спада: {slope_low:.1f} дБ/октава")
        
        print(f"   Верхняя граница:")
        print(f"     • f_high = {f_high_3db:.0f} МГц: {amp_high_3db:.1f} дБ")
        print(f"     • f_high×2 = {f_high_double:.0f} МГц: {amp_high_double:.1f} дБ")
        print(f"     • Крутизна спада: {slope_high:.1f} дБ/октава")
        
        print(f"\n📈 ДОПОЛНИТЕЛЬНО:")
        print(f"   • Fs:                     {Fs:>8.0f} МГц")
        print(f"   • Отсчетов:               {len(t):>8,}")
        print(f"   • Максимум спектра:       {max_amp_db:>8.1f} дБ")
    
    # Интерфейс
    interact(plot_lfm,
             T=FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1, 
                          description='Длительность (мс):'),
             f_start=IntSlider(value=1000, min=100, max=7000, step=100, 
                              description='f_start (МГц):'),
             f_end=IntSlider(value=2000, min=200, max=10000, step=100, 
                            description='f_end (МГц):'))

# Запуск
print("🎛️ ПРОДВИНУТЫЙ АНАЛИЗ ЛЧМ СИГНАЛА")
print("📊 Коэффициент ЛЧМ, девиация, крутизна спада")
advanced_positive_lfm_db()

🎛️ ПРОДВИНУТЫЙ АНАЛИЗ ЛЧМ СИГНАЛА
📊 Коэффициент ЛЧМ, девиация, крутизна спада


interactive(children=(FloatSlider(value=1.0, description='Длительность (мс):', max=10.0, min=0.1), IntSlider(v…